In [8]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"C:\Plegma_Programming")
INPUT_DIR = BASE_DIR / "Processed_Houses_Final"

CHECK_HOUSES = ["House_01", "House_04", "House_11"]

THRESHOLD = 5  # μόνο blocks >5 διορθώνονται

for house in CHECK_HOUSES:
    file_path = INPUT_DIR / f"{house}.csv"
    df = pd.read_csv(file_path)
    
    print(f"\nProcessing {house}...")
    
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")
    
    # detect zero blocks
    df["is_zero"] = df["energy_Wh"] == 0
    df["group"] = (df["is_zero"] != df["is_zero"].shift()).cumsum()
    
    groups = df[df["is_zero"]].groupby("group").agg(
        start=("timestamp", "first"),
        end=("timestamp", "last"),
        length=("timestamp", "count")
    )
    
    # mark rows to interpolate
    df["fix_flag"] = False
    
    for g, row in groups.iterrows():
        if row["length"] > THRESHOLD:
            df.loc[df["group"] == g, "fix_flag"] = True
            print(f"Fixing block: {row['start']} → {row['end']} (len={row['length']})")
    
    # replace only flagged zeros with NaN
    df.loc[df["fix_flag"], "energy_Wh"] = None
    
    # interpolate
    df["energy_Wh"] = df["energy_Wh"].interpolate(method="linear")
    
    # fallback (edges)
    df["energy_Wh"] = df["energy_Wh"].fillna(method="ffill").fillna(method="bfill")
    
    # cleanup
    df = df.drop(columns=["is_zero", "group", "fix_flag"])
    
    # SAVE (overwrite)
    df.to_csv(file_path, index=False)
    
    print("Done.")


Processing House_01...
Fixing block: 2022-08-31 00:30:00 → 2022-08-31 23:30:00 (len=47)


C:\Users\z0050azt\AppData\Local\Temp\ipykernel_5052\2342548059.py:45: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["energy_Wh"] = df["energy_Wh"].fillna(method="ffill").fillna(method="bfill")


Done.

Processing House_04...


C:\Users\z0050azt\AppData\Local\Temp\ipykernel_5052\2342548059.py:45: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["energy_Wh"] = df["energy_Wh"].fillna(method="ffill").fillna(method="bfill")


Done.

Processing House_11...
Fixing block: 2023-06-30 12:00:00 → 2023-07-01 06:30:00 (len=38)
Done.


C:\Users\z0050azt\AppData\Local\Temp\ipykernel_5052\2342548059.py:45: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["energy_Wh"] = df["energy_Wh"].fillna(method="ffill").fillna(method="bfill")
